In [ ]:
import numpy

In [ ]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 29.9 MB/s eta 0:00:00


In [ ]:
import torch
torch.cuda.is_available()

True

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import io
import random
from pathlib  import Path
import requests
from PIL import Image
import os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models
from torchvision.models import ResNet18_Weights
print(torch.__version__)

2.11.0+cu128


In [ ]:
train_dir = Path(r"/content/drive/MyDrive/Car_Demage_Severity/training")
test_dir = Path(r"/content/drive/MyDrive/Car_Demage_Severity/validation")
save_dir = Path("/content/drive/MyDrive/yolov8_train_car/pkl")
save_dir.mkdir(parents=True, exist_ok=True)

model_path = save_dir / 'cnn_car.pkl'
seed=42
random.seed(seed)
torch.manual_seed(seed)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device


device(type='cuda')

In [ ]:
import torch.nn as nn
from torchvision import transforms
from torchvision.models import resnet50, ResNet50_Weights

# 1. Khởi tạo mô hình ResNet50 và lấy weights mặc định
weights = ResNet50_Weights.DEFAULT
model = resnet50(weights=weights)

# Tự động lấy mean và std chuẩn từ pre-trained weights của ResNet50
base_eval_transform = weights.transforms()
mean = base_eval_transform.mean
std = base_eval_transform.std

# 2. Định nghĩa kích thước ảnh đồng bộ cho ResNet50 (Dùng 224 hoặc 256)
image_size = 224
resize_size = 256

# 3. Cấu hình lớp Classifier Head (ResNet50 sử dụng biến .fc)
# Số lượng in_features của ResNet50 là 2048 (lớn hơn hẳn ResNet18 là 512)
num_ftrs = model.fc.in_features
model.fc = nn.Sequential(
    nn.Dropout(p=0.40), # Thêm dropout chống overfitting
    nn.Linear(num_ftrs,3)
)
model = model.to(device)

# 4. Cấu hình Eval Transform
eval_transform = transforms.Compose([
    transforms.Resize((resize_size, resize_size)),
    transforms.CenterCrop((image_size, image_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std),
])

# 5. Cấu hình Train Transform
train_transform = transforms.Compose([
     transforms.Resize((resize_size, resize_size)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomApply([
        transforms.ColorJitter(
            brightness=0.12,
            contrast=0.12,
            saturation=0.08,
            hue=0.03,
        )
    ], p=0.5),
    transforms.RandomAffine(
        degrees=5,
        translate=(0.03, 0.03),
        scale=(0.95, 1.05),
    ),
    transforms.CenterCrop((image_size, image_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std),
])

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 161MB/s]


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score

full_train_for_train = datasets.ImageFolder(root=str(train_dir), transform=train_transform)
full_train_for_eval = datasets.ImageFolder(root=str(train_dir), transform=eval_transform)
test_dataset = datasets.ImageFolder(root=str(test_dir), transform=eval_transform)
import numpy as np
targets = np.array(full_train_for_train.targets)
indices = np.arange(len(targets))

train_idx, val_idx = train_test_split(
    indices,
    test_size=0.25,
    random_state=seed,
    stratify=targets,
)

train_dataset = Subset(full_train_for_train, train_idx)
val_dataset = Subset(full_train_for_eval, val_idx)

class_names = full_train_for_train.classes
print("Classes:", class_names)
print(f"Train/Val/Test: {len(train_dataset)}/{len(val_dataset)}/{len(test_dataset)}")

Classes: ['01-minor', '02-moderate', '03-severe']
Train/Val/Test: 2209/737/390


In [ ]:
# Khởi tạo DataLoader cho tập Train, Validation và Test
batch_size = 32  # Bạn có thể điều chỉnh kích thước batch tùy ý

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print(f"Đã khởi tạo các DataLoader thành công!")

Đã khởi tạo các DataLoader thành công!


In [ ]:
train_targets = targets[train_idx]
class_counts = np.bincount(train_targets, minlength=len(class_names))

class_weights = class_counts.sum() / (len(class_counts) * np.maximum(class_counts, 1))
class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)

criterion = nn.CrossEntropyLoss(
    weight=class_weights,
    label_smoothing=0.05,
)

print("Class counts:", dict(zip(class_names, class_counts)))
print("Class weights:", class_weights)

Class counts: {'01-minor': np.int64(807), '02-moderate': np.int64(693), '03-severe': np.int64(709)}
Class weights: tensor([0.9124, 1.0625, 1.0386], device='cuda:0')


In [ ]:
@torch.no_grad()
def evaluate(model, loader, device, criterion=None):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_samples = 0
    all_true = []
    all_pred = []

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            logits = model(images)

            if criterion is not None:
                loss = criterion(logits, labels)
                total_loss += loss.item() * labels.size(0)

            preds = logits.argmax(dim=1)
            total_correct += (preds == labels).sum().item()
            total_samples += labels.size(0)

            all_true.extend(labels.cpu().numpy())
            all_pred.extend(preds.cpu().numpy())

    return {
        "loss": total_loss / max(total_samples, 1),
        "accuracy": total_correct / max(total_samples, 1),
        "macro_f1": f1_score(all_true, all_pred, average="macro"),
        "y_true": all_true,
        "y_pred": all_pred,
    }


def train_one_phase(model, train_loader, val_loader, device, epochs, optimizer, save_path):
    history = []
    best_val_acc = 0.0

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        running_correct = 0
        total_samples = 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            logits = model(images)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * labels.size(0)
            running_correct += (logits.argmax(dim=1) == labels).sum().item()
            total_samples += labels.size(0)

        train_metrics = {
            'loss': running_loss / max(total_samples, 1),
            'accuracy': running_correct / max(total_samples, 1),
        }
        val_metrics = evaluate(model, val_loader, device)

        if val_metrics['accuracy'] > best_val_acc:
            best_val_acc = val_metrics['accuracy']
            torch.save(model.state_dict(), save_path)

        history.append({'epoch': epoch, 'train': train_metrics, 'val': val_metrics})
        print(
            f"Epoch {epoch:02d} | train_loss={train_metrics['loss']:.4f} train_acc={train_metrics['accuracy']:.4f} | "
            f"val_loss={val_metrics['loss']:.4f} val_acc={val_metrics['accuracy']:.4f}"
        )

    print(f'Best val_acc: {best_val_acc:.4f}')
    return history

In [ ]:


# Build transfer learning model
# Sửa models.resnet18 thành models.resnet50
model = models.resnet50(weights=weights).to(device)
num_ftrs = model.fc.in_features
model.fc = nn.Sequential(
    nn.Dropout(p=0.35),
    nn.Linear(num_ftrs, 3),
).to(device)

# Phase 1: freeze backbone, train only classifier head
# Phase 1: Đóng băng toàn bộ các layer tính toán đặc trưng
for param in model.parameters():
    param.requires_grad = False

# Chỉ mở khóa duy nhất lớp phân loại cuối cùng (fc) để huấn luyện head
for param in model.fc.parameters():
    param.requires_grad = True

optimizer_phase1 = torch.optim.AdamW(
    model.fc.parameters(),
    lr=3e-4,
    weight_decay=1e-4,
)
print('Phase 1: train classifier head')
history_phase1 = train_one_phase(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    epochs=25,
    optimizer=optimizer_phase1,
    save_path=model_path,
)

Phase 1: train classifier head
Epoch 01 | train_loss=0.9666 train_acc=0.6057 | val_loss=0.0000 val_acc=0.6988
Epoch 02 | train_loss=0.8072 train_acc=0.7134 | val_loss=0.0000 val_acc=0.7259
Epoch 03 | train_loss=0.7376 train_acc=0.7451 | val_loss=0.0000 val_acc=0.7544
Epoch 04 | train_loss=0.7040 train_acc=0.7583 | val_loss=0.0000 val_acc=0.7531
Epoch 05 | train_loss=0.6712 train_acc=0.7610 | val_loss=0.0000 val_acc=0.7680
Epoch 06 | train_loss=0.6620 train_acc=0.7732 | val_loss=0.0000 val_acc=0.7544
Epoch 07 | train_loss=0.6393 train_acc=0.7795 | val_loss=0.0000 val_acc=0.7612
Epoch 08 | train_loss=0.6285 train_acc=0.7850 | val_loss=0.0000 val_acc=0.7639
Epoch 09 | train_loss=0.6169 train_acc=0.7872 | val_loss=0.0000 val_acc=0.7693
Epoch 10 | train_loss=0.6086 train_acc=0.7927 | val_loss=0.0000 val_acc=0.7775
Epoch 11 | train_loss=0.6116 train_acc=0.7845 | val_loss=0.0000 val_acc=0.7829
Epoch 12 | train_loss=0.5992 train_acc=0.7995 | val_loss=0.0000 val_acc=0.7734
Epoch 13 | train_loss

In [ ]:
# Phase 2: fine-tune layer4 + fc
print('Phase 2: fine-tune layer4 + fc của ResNet50')

# 1. Khóa tất cả các layer lại trước
for param in model.parameters():
    param.requires_grad = False

# 2. Mở khóa layer4 (các block tích chập cuối cùng) và fc head
for param in model.layer4.parameters():
    param.requires_grad = True
for param in model.fc.parameters():
    param.requires_grad = True

# 3. Thiết lập Optimizer với Learning Rate rất nhỏ cho layer4 để tránh làm hỏng weights
optimizer_phase2 = torch.optim.AdamW([
    {'params': model.layer4.parameters(), 'lr': 5e-6}, # Giảm xuống 5e-6 thay vì 1e-5
    {'params': model.fc.parameters(), 'lr': 3e-5},     # Giảm xuống 3e-5 thay vì 5e-5
], weight_decay=1e-4)

print('Phase 2: fine-tune layer4 + fc')
# 2. Giảm số lượng epoch tối đa xuống 20 để tránh Overfitting quá đà
history_phase2 = train_one_phase(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    epochs=20, # <--- Sửa từ 50 xuống 20
    optimizer=optimizer_phase2,
    save_path=model_path,
)
# Load best checkpoint and evaluate test
model.load_state_dict(torch.load(model_path, map_location=device))
test_metrics = evaluate(model, test_loader, device, criterion)

print("Test accuracy:", test_metrics["accuracy"])
print("Test macro_f1:", test_metrics["macro_f1"])

print(classification_report(
    test_metrics["y_true"],
    test_metrics["y_pred"],
    target_names=class_names,
))

print(confusion_matrix(
    test_metrics["y_true"],
    test_metrics["y_pred"],
))

Phase 2: fine-tune layer4 + fc của ResNet50
Phase 2: fine-tune layer4 + fc
Epoch 01 | train_loss=0.5503 train_acc=0.8257 | val_loss=0.0000 val_acc=0.7910
Epoch 02 | train_loss=0.5545 train_acc=0.8158 | val_loss=0.0000 val_acc=0.7978
Epoch 03 | train_loss=0.5448 train_acc=0.8153 | val_loss=0.0000 val_acc=0.7978
Epoch 04 | train_loss=0.5211 train_acc=0.8343 | val_loss=0.0000 val_acc=0.8060
Epoch 05 | train_loss=0.5321 train_acc=0.8307 | val_loss=0.0000 val_acc=0.8100
Epoch 06 | train_loss=0.5166 train_acc=0.8330 | val_loss=0.0000 val_acc=0.8033
Epoch 07 | train_loss=0.5208 train_acc=0.8407 | val_loss=0.0000 val_acc=0.7978
Epoch 08 | train_loss=0.5097 train_acc=0.8388 | val_loss=0.0000 val_acc=0.8060
Epoch 09 | train_loss=0.5023 train_acc=0.8434 | val_loss=0.0000 val_acc=0.8005
Epoch 10 | train_loss=0.5032 train_acc=0.8339 | val_loss=0.0000 val_acc=0.8033
Epoch 11 | train_loss=0.4940 train_acc=0.8497 | val_loss=0.0000 val_acc=0.8073
Epoch 12 | train_loss=0.4933 train_acc=0.8470 | val_loss

In [ ]:
# Predict from image URL
def predict_from_url(url: str):
    resp = requests.get(url, timeout=20)
    resp.raise_for_status()

    image = Image.open(io.BytesIO(resp.content)).convert('RGB')
    x = eval_transform(image).unsqueeze(0).to(device)

    model.eval()
    with torch.no_grad():
        logits = model(x)
        probs = torch.softmax(logits, dim=1)[0].cpu()

    idx = int(torch.argmax(probs).item())
    return {
        'label': class_names[idx],
        'confidence': float(probs[idx].item()),
        'probs': {class_names[i]: float(probs[i].item()) for i in range(len(class_names))},
    }

In [ ]:
# Example URL prediction
import cv2
sample_url = 'https://tamanhhospital.vn/wp-content/uploads/2024/06/hinh-anh-nhan-biet-kien-ba-khoang.jpg'
result = predict_from_url(sample_url)
print(result)

{'label': '03-severe', 'confidence': 0.3704225420951843, 'probs': {'01-minor': 0.32932034134864807, '02-moderate': 0.30025714635849, '03-severe': 0.3704225420951843}}
